In [1]:
from pyspark.sql import SparkSession

# packages for iceberg + spark + kafka
PACKAGES = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "org.postgresql:postgresql:42.6.0",
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
]

spark = SparkSession.builder \
    .appName("Iceberg_Setup") \
    .config("spark.jars.packages", ",".join(PACKAGES)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
    .config("spark.sql.catalog.my_catalog.uri", "jdbc:postgresql://postgres:5432/iceberg_metastore") \
    .config("spark.sql.catalog.my_catalog.jdbc.user", "iceberg") \
    .config("spark.sql.catalog.my_catalog.jdbc.password", "iceberg") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
print('123')

123


In [24]:
# create default namespace and table
spark.sql("CREATE NAMESPACE IF NOT EXISTS my_catalog.default")
# 1. create unsorted table
spark.sql("DROP TABLE IF EXISTS my_catalog.default.user_events_unsorted")
spark.sql("""
CREATE TABLE my_catalog.default.user_events_unsorted (
    user_id INT,
    shop STRING,    
    event_type STRING,
    device_type STRING,
    price DOUBLE,
    session_id STRING,
    timestamp TIMESTAMP
)
USING iceberg
TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
)
""")
print("unsorted table created")

# 2. create sorted table
spark.sql("DROP TABLE IF EXISTS my_catalog.default.user_events_sorted")
spark.sql("""
CREATE TABLE my_catalog.default.user_events_sorted (
    user_id INT,
    shop STRING,    
    event_type STRING,
    device_type STRING,
    price DOUBLE,
    session_id STRING,
    timestamp TIMESTAMP
)
USING iceberg
TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
)
""")

#  enforce sorting before write
spark.sql("ALTER TABLE my_catalog.default.user_events_sorted WRITE ORDERED BY shop, device_type, event_type")
print("sorted table created and configured")

spark.sql("DESCRIBE my_catalog.default.user_events_sorted").show()
spark.sql("DESCRIBE my_catalog.default.user_events_unsorted").show()

unsorted table created
sorted table created and configured
+-----------+---------+-------+
|   col_name|data_type|comment|
+-----------+---------+-------+
|    user_id|      int|   NULL|
|       shop|   string|   NULL|
| event_type|   string|   NULL|
|device_type|   string|   NULL|
|      price|   double|   NULL|
| session_id|   string|   NULL|
|  timestamp|timestamp|   NULL|
+-----------+---------+-------+

+-----------+---------+-------+
|   col_name|data_type|comment|
+-----------+---------+-------+
|    user_id|      int|   NULL|
|       shop|   string|   NULL|
| event_type|   string|   NULL|
|device_type|   string|   NULL|
|      price|   double|   NULL|
| session_id|   string|   NULL|
|  timestamp|timestamp|   NULL|
+-----------+---------+-------+



In [25]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col, from_json

# 1. schema
json_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("shop", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("session_id", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

# 2. read from kafka (once)
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "user_events") \
    .option("startingOffsets", "latest") \
    .load()

# 3. parse data
parsed_stream = kafka_stream \
    .selectExpr("CAST(value AS STRING) as json_string") \
    .select(from_json(col("json_string"), json_schema).alias("data")) \
    .select("data.*")

print("launching both write streams...")

# 4. stream 1: write to regular table (unsorted)
streaming_query_unsorted = parsed_stream.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .trigger(processingTime="3 seconds") \
    .option("checkpointLocation", "s3a://warehouse/checkpoints/unsorted_v2") \
    .toTable("my_catalog.default.user_events_unsorted")

# 5. stream 2: write to sorted table
streaming_query_sorted = parsed_stream.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .trigger(processingTime="3 seconds") \
    .option("checkpointLocation", "s3a://warehouse/checkpoints/sorted_v2") \
    .toTable("my_catalog.default.user_events_sorted")

print("streams are running in the background")

launching both write streams...
streams are running in the background


In [28]:
import time
import os
from datetime import datetime

LOG_STREAM_FILE = "streaming_write_metrics.csv"

# create a file for write metrics if it does not exist
if not os.path.exists(LOG_STREAM_FILE):
    with open(LOG_STREAM_FILE, "w") as f:
        f.write("timestamp,rows_unsorted,duration_ms_unsorted,rows_sorted,duration_ms_sorted\n")
    print(f"created log file for streaming: {LOG_STREAM_FILE}")

print("starting micro-batch metrics collection (press stop to halt)...")

try:
    while True:
        # get the latest progress (these are dictionaries in json format)
        prog_unsorted = streaming_query_unsorted.lastProgress
        prog_sorted = streaming_query_sorted.lastProgress
        
        # record only if both streams have processed at least once
        if prog_unsorted and prog_sorted:
            current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            
            # extract the number of rows and duration (milliseconds)
            rows_unsorted = prog_unsorted.get('numInputRows', 0)
            dur_unsorted = prog_unsorted.get('batchDuration', 0)
            
            rows_sorted = prog_sorted.get('numInputRows', 0)
            dur_sorted = prog_sorted.get('batchDuration', 0)
            
            # if rows are 0 (no data), we can skip recording or record for completeness
            if rows_unsorted > 0 or rows_sorted > 0:
                with open(LOG_STREAM_FILE, "a") as f:
                    f.write(f"{current_time},{rows_unsorted},{dur_unsorted},{rows_sorted},{dur_sorted}\n")
                
                print(f"[{current_time}] batch: unsorted={dur_unsorted}ms ({rows_unsorted} rows) | sorted={dur_sorted}ms ({rows_sorted} rows)")
        
        # poll every 3 seconds
        time.sleep(10) 
except KeyboardInterrupt:
    print("\nmetrics collection stopped.")

created log file for streaming: streaming_write_metrics.csv
starting micro-batch metrics collection (press stop to halt)...
[2026-05-20 21:30:48] batch: unsorted=0ms (1021 rows) | sorted=0ms (1620 rows)
[2026-05-20 21:30:58] batch: unsorted=0ms (993 rows) | sorted=0ms (2104 rows)
[2026-05-20 21:31:08] batch: unsorted=0ms (982 rows) | sorted=0ms (1962 rows)
[2026-05-20 21:31:19] batch: unsorted=0ms (1076 rows) | sorted=0ms (2198 rows)
[2026-05-20 21:31:29] batch: unsorted=0ms (1112 rows) | sorted=0ms (2224 rows)
[2026-05-20 21:31:39] batch: unsorted=0ms (886 rows) | sorted=0ms (1770 rows)
[2026-05-20 21:31:49] batch: unsorted=0ms (752 rows) | sorted=0ms (1854 rows)
[2026-05-20 21:31:59] batch: unsorted=0ms (869 rows) | sorted=0ms (1598 rows)
[2026-05-20 21:32:09] batch: unsorted=0ms (670 rows) | sorted=0ms (1338 rows)
[2026-05-20 21:32:19] batch: unsorted=0ms (615 rows) | sorted=0ms (1424 rows)
[2026-05-20 21:32:29] batch: unsorted=0ms (508 rows) | sorted=0ms (1612 rows)
[2026-05-20 21:

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


Py4JError: An error occurred while calling o231.lastProgress

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [26]:
spark.sql("SELECT * FROM my_catalog.default.user_events_unsorted ORDER BY timestamp DESC LIMIT 5").show()

+-------+-------+----------------+-----------+------+--------------------+--------------------+
|user_id|   shop|      event_type|device_type| price|          session_id|           timestamp|
+-------+-------+----------------+-----------+------+--------------------+--------------------+
|   1550|foxtrot|       page_view|    desktop|113.81|004aea74-fa73-48c...|2026-05-20 21:30:...|
|   5322| citrus|register_success|    desktop|   0.0|33e2d3e3-2ff1-479...|2026-05-20 21:30:...|
|   8270|foxtrot|     change_lang| mobile_web|   0.0|09980601-852b-426...|2026-05-20 21:30:...|
|    992|rozetka|        purchase| mobile_web|   0.0|9e23a350-25f1-487...|2026-05-20 21:30:...|
|   5477| citrus|register_success|        ios|   0.0|b2adc3b8-0051-4fd...|2026-05-20 21:30:...|
+-------+-------+----------------+-----------+------+--------------------+--------------------+



In [27]:
spark.sql("SELECT * FROM my_catalog.default.user_events_sorted ORDER BY timestamp DESC LIMIT 5").show()

+-------+-------+----------------+-----------+------+--------------------+--------------------+
|user_id|   shop|      event_type|device_type| price|          session_id|           timestamp|
+-------+-------+----------------+-----------+------+--------------------+--------------------+
|   1550|foxtrot|       page_view|    desktop|113.81|004aea74-fa73-48c...|2026-05-20 21:30:...|
|   5322| citrus|register_success|    desktop|   0.0|33e2d3e3-2ff1-479...|2026-05-20 21:30:...|
|   8270|foxtrot|     change_lang| mobile_web|   0.0|09980601-852b-426...|2026-05-20 21:30:...|
|    992|rozetka|        purchase| mobile_web|   0.0|9e23a350-25f1-487...|2026-05-20 21:30:...|
|   5477| citrus|register_success|        ios|   0.0|b2adc3b8-0051-4fd...|2026-05-20 21:30:...|
+-------+-------+----------------+-----------+------+--------------------+--------------------+

